In [1]:
# bibliotecques
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Chemin d'accès 
ROOT = Path.cwd().parent 
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
print(f"Project root: {ROOT}")

from src.data_loader import *
from src.periodicity import *
from src.hull_white import *
from src.sinusoidal_hw import *
from src.monte_carlo import *
from src.calibration import *
print("Imports OK")

Project root: C:\Users\Manuel\Desktop\sinusoidal_hw
Imports OK


In [3]:
# 1. Chargement des données
data_path = ROOT / 'data' / 'fredgraph.csv'
loader = YieldDataLoader(str(data_path))
df = loader.load_data()

Données chargées. 1292 lignes supprimées (NaN/Jours fériés).
Période: 1993-10-01 à 2022-12-30


In [19]:
# 2. Sélection de la courbe de taux à calibrer
# Le papier mentionne la période finissant en Dec 2022. 
# Nous calibrons sur la DERNIÈRE date disponible pour avoir l'état "actuel" du marché.

calibration_date = df.index[-2]
current_curve = df.loc[calibration_date]

print(f"--- Calibration sur la courbe du {calibration_date.date()} ---")
print(current_curve)

# Préparation des inputs
# Mapping des maturités (ex: '1Y' -> 1.0)
maturities_map = {'1Y': 1.0, '2Y': 2.0, '3Y': 3.0, '5Y': 5.0, '7Y': 7.0, '10Y': 10.0, '20Y': 20.0, '30Y': 30.0}
maturities = []
yields = []

for tenor, years in maturities_map.items():
    if tenor in current_curve:
        maturities.append(years)
        yields.append(current_curve[tenor]) # Déjà en décimales via data_loader

r0 = yields[0] # Le taux court initial est approximé par le taux 1 an (1.22% dans le papier)
print(f"Taux court initial (r0) : {r0:.4%}")


--- Calibration sur la courbe du 2022-12-29 ---
1Y     0.0471
2Y     0.0434
3Y     0.0416
5Y     0.0394
7Y     0.0391
10Y    0.0383
20Y    0.0409
30Y    0.0392
Name: 2022-12-29 00:00:00, dtype: float64
Taux court initial (r0) : 4.7100%


In [21]:
# 3. Initialisation du Calibrateur
calibrator = Calibrator(maturities, yields)

In [23]:
# 4. Calibration Hull-White Standard
res_hw = calibrator.calibrate_standard_hw(r0)
print("\n--- Résultats Calibration HW Standard ---")
print(res_hw)


--- Résultats Calibration HW Standard ---
{'kappa': 1.988224485871627, 'theta': 0.039473833744052986, 'sigma': 0.030076061577016748, 'rmse': 0.0062775027810458685, 'success': True}


In [25]:
# 5. Calibration Sinusoidal HW
# IMPORTANT : On utilise le OMEGA calculé en Phase 2
# Remplacez cette valeur par celle trouvée dans le notebook 2 si différente

FINAL_OMEGA_FROM_PHASE_2 = 0.000859
res_sin = calibrator.calibrate_sinusoidal_hw(r0, fixed_omega=FINAL_OMEGA_FROM_PHASE_2)
print("\n--- Résultats Calibration Sinusoidal HW ---")
print(res_sin)

Début calibration Sinusoidal HW (Monte Carlo, omega=0.000859)...

--- Résultats Calibration Sinusoidal HW ---
{'kappa_0': 0.26489665036249105, 'A': 0.213832754100864, 'omega': 0.000859, 'theta': 0.03891462971755882, 'sigma': 0.008966547722105263, 'rmse': 0.008852056295628427, 'success': False}


In [26]:
# 6. Comparaison RMSE
print("\n--- Comparaison RMSE ---")
print(f"Standard HW   : {res_hw['rmse']:.6f}")
print(f"Sinusoidal HW : {res_sin['rmse']:.6f}")


--- Comparaison RMSE ---
Standard HW   : 0.006278
Sinusoidal HW : 0.008852
